In [1]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from scipy.special import logit
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
TARGET = "is_fraud"
ID_COL = "id"
OOF_DIR = Path("oof")
SUB_DIR = Path("submissions")

comp_path = Path(kagglehub.competition_download("playground-series-s6e11"))
train = pd.read_csv(comp_path / "train.csv", usecols=[ID_COL, TARGET])
test = pd.read_csv(comp_path / "test.csv", usecols=[ID_COL])
sample_sub = pd.read_csv(comp_path / "sample_submission.csv")

y = train[TARGET].astype(int).reset_index(drop=True)
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))

def rank01(v):
    return (rankdata(v) - 0.5) / len(v)

def logit_rank(v):
    return logit(rank01(v))

def load_mine(name):
    return np.load(OOF_DIR / f"{name}_oof.npy"), np.load(OOF_DIR / f"{name}_test.npy")

oof_lgb, _ = load_mine("lgb_glm")
print("train", y.shape, "| test", test.shape, "| folds", len(FOLDS))
print(f"Check lgb_glm OOF AUC: {roc_auc_score(y, oof_lgb):.5f} (should be 0.95056)")

train (699960,) | test (299984, 1) | folds 5
Check lgb_glm OOF AUC: 0.95056 (should be 0.95056)


In [2]:
art_path = Path(kagglehub.dataset_download("goodpjw2008/s6e11-fraud-series-artifacts"))
files = sorted(p.relative_to(art_path).as_posix() for p in art_path.rglob("*") if p.is_file())
print(len(files), "files")
for f in files:
    print(f)

100%|██████████| 129M/129M [00:05<00:00, 24.9MB/s] 

Extracting files...


68 files
ebm_b4_oof.npy
ebm_b4_test.npy
ebm_dig_full.npy
ebm_dig_oof.npy
ebm_dig_test.npy
ebm_i25_bag4_full.npy
ebm_i25_bag4_oof.npy
gb_bag5_full.npy
gb_bag5_oof.npy
gb_s0_full.npy
gb_s0_glmoof.npy
gb_s0_oof.npy
gb_s1_full.npy
gb_s1_glmoof.npy
gb_s1_oof.npy
gb_s2_full.npy
gb_s2_glmoof.npy
gb_s2_oof.npy
gb_s3_full.npy
gb_s3_glmoof.npy
gb_s3_oof.npy
gb_s4_full.npy
gb_s4_glmoof.npy
gb_s4_oof.npy
gb_testbag_full.npy
gbn_bag5_w128_full.npy
gbn_bag5_w128_oof.npy
gbn_bag5_w512_full.npy
gbn_bag5_w512_oof.npy
gbn_w128_testbag_full.npy
gbn_w512_testbag_full.npy
gbx_bag3_full.npy
gbx_bag3_oof.npy
glm_dl_s50_bag_full.npy
glm_dl_s50_bag_oof.npy
glm_dl_s50_oof.npy
glm_ohk2_bag_full.npy
glm_ohk2_bag_oof.npy
glm_ohk_c01_full.npy
glm_ohk_c01_oof.npy
gpt2-tokenizer.json
lgb_dig_l7_full.npy
lgb_dig_l7_oof.npy
lgb_dig_s100_bag5_full.npy
lgb_dig_s100_bag5_oof.npy
lgb_dig_s100_bag_full.npy
lgb_dig_s100_bag_oof.npy
lgb_diglead_bag_full.npy
lgb_diglead_bag_oof.npy
lgb_diglead_l3_bag_full.npy
lgb_diglead_l3_ba

In [3]:
from scipy.stats import spearmanr

wide_oof = np.load(art_path / "wide_stack_nested.npy").ravel()
wide_test = np.load(art_path / "wide_stack_test.npy").ravel()
wide_lm_oof = np.load(art_path / "wide_lm_stack_nested.npy").ravel()

print("wide_stack:", wide_oof.shape, wide_test.shape)
print(f"wide_stack nested CV:    {roc_auc_score(y, wide_oof):.5f} (should be about 0.9525)")
print(f"wide_lm_stack nested CV: {roc_auc_score(y, wide_lm_oof):.5f}")

MINE = ["lgb_glm", "xgb_keys", "cat_dig", "glm_keys", "lgb_keys", "cat_dig_te", "lgb_dig_l4"]
mine_oof = {name: load_mine(name)[0] for name in MINE}
v7_oof = 0.4 * rank01(mine_oof["lgb_glm"]) + 0.4 * rank01(mine_oof["xgb_keys"]) + 0.2 * rank01(mine_oof["cat_dig"])

print(f"\nmy v7 blend CV: {roc_auc_score(y, v7_oof):.5f}")
print(f"rank correlation, v7 vs wide_stack: {spearmanr(v7_oof, wide_oof)[0]:.4f}")
for name in MINE:
    print(f"  {name:12s} CV {roc_auc_score(y, mine_oof[name]):.5f} | corr with wide_stack {spearmanr(mine_oof[name], wide_oof)[0]:.4f}")

wide_stack: (699960,) (299984,)
wide_stack nested CV:    0.95250 (should be about 0.9525)
wide_lm_stack nested CV: 0.95257

my v7 blend CV: 0.95067
rank correlation, v7 vs wide_stack: 0.9854
  lgb_glm      CV 0.95056 | corr with wide_stack 0.9838
  xgb_keys     CV 0.95004 | corr with wide_stack 0.9802
  cat_dig      CV 0.94777 | corr with wide_stack 0.9582
  glm_keys     CV 0.94913 | corr with wide_stack 0.9745
  lgb_keys     CV 0.95003 | corr with wide_stack 0.9798
  cat_dig_te   CV 0.94844 | corr with wide_stack 0.9668
  lgb_dig_l4   CV 0.94855 | corr with wide_stack 0.9671


In [4]:
WEIGHTS = np.round(np.arange(0, 0.51, 0.05), 2)

def two_way(base, extra, w):
    return (1 - w) * logit_rank(base) + w * logit_rank(extra)

def nested_weight(base_oof, extra_oof):
    blended = np.zeros(len(y))
    chosen = []
    for train_idx, valid_idx in FOLDS:
        scores = [roc_auc_score(y.iloc[train_idx], two_way(base_oof[train_idx], extra_oof[train_idx], w))
                  for w in WEIGHTS]
        w = float(WEIGHTS[int(np.argmax(scores))])
        chosen.append(w)
        blended[valid_idx] = rank01(two_way(base_oof[valid_idx], extra_oof[valid_idx], w))
    return roc_auc_score(y, blended), chosen

candidates = {name: mine_oof[name] for name in MINE}
candidates["my v7 blend"] = v7_oof

base_cv = roc_auc_score(y, wide_oof)
print(f"wide_stack alone: {base_cv:.5f}\n")
for label, extra in candidates.items():
    score, chosen = nested_weight(wide_oof, extra)
    print(f"+ {label:12s} nested CV {score:.5f} ({score - base_cv:+.5f}) | weight per fold {chosen}")

wide_stack alone: 0.95250

+ lgb_glm      nested CV 0.95248 (-0.00002) | weight per fold [0.05, 0.0, 0.05, 0.0, 0.1]
+ xgb_keys     nested CV 0.95249 (-0.00001) | weight per fold [0.0, 0.0, 0.0, 0.0, 0.05]
+ cat_dig      nested CV 0.95250 (+0.00000) | weight per fold [0.0, 0.0, 0.0, 0.0, 0.0]
+ glm_keys     nested CV 0.95248 (-0.00003) | weight per fold [0.05, 0.0, 0.05, 0.0, 0.1]
+ lgb_keys     nested CV 0.95247 (-0.00003) | weight per fold [0.0, 0.0, 0.0, 0.0, 0.1]
+ cat_dig_te   nested CV 0.95250 (+0.00000) | weight per fold [0.0, 0.0, 0.0, 0.0, 0.0]
+ lgb_dig_l4   nested CV 0.95249 (-0.00002) | weight per fold [0.05, 0.0, 0.0, 0.05, 0.05]
+ my v7 blend  nested CV 0.95248 (-0.00003) | weight per fold [0.05, 0.0, 0.05, 0.0, 0.1]


In [5]:
wide_lm_test = np.load(art_path / "wide_lm_stack_test.npy").ravel()

score, chosen = nested_weight(wide_oof, wide_lm_oof)
print(f"wide_stack alone:         {roc_auc_score(y, wide_oof):.5f}")
print(f"wide_lm_stack alone:      {roc_auc_score(y, wide_lm_oof):.5f}")
print(f"mix, nested:              {score:.5f} | weight on wide_lm per fold {chosen}")

w = float(np.mean(chosen))
pred = rank01(two_way(wide_test, wide_lm_test, w))
submission = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), TARGET: pred})
submission = submission.set_index(ID_COL).loc[sample_sub[ID_COL]].reset_index()
submission.to_csv(SUB_DIR / "public_stack.csv", index=False)
print(f"Saved public_stack.csv with weight {w:.2f} on wide_lm_stack", submission.shape)

wide_stack alone:         0.95250
wide_lm_stack alone:      0.95257
mix, nested:              0.95256 | weight on wide_lm per fold [0.5, 0.5, 0.5, 0.5, 0.5]
Saved public_stack.csv with weight 0.50 on wide_lm_stack (299984, 2)


In [6]:
orig_path = Path(kagglehub.dataset_download("uditjain13/credit-card-fraud-detection-2026"))
print(orig_path / "credit_card_fraud_2026.csv")

C:\Users\user\.cache\kagglehub\datasets\uditjain13\credit-card-fraud-detection-2026\versions\5\credit_card_fraud_2026.csv


In [7]:
import shutil
from pathlib import Path

src = Path(r"C:\Users\user\.cache\kagglehub\datasets\uditjain13\credit-card-fraud-detection-2026\versions\5\credit_card_fraud_2026.csv")
dst = Path.home() / "Desktop" / "credit_card_fraud_2026.csv"
shutil.copy2(src, dst)
print("Copied to", dst)

Copied to C:\Users\user\Desktop\credit_card_fraud_2026.csv
